In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import polars as pl
import duckdb
import glob
import duckdb
import gc
import os


#### Carregando os dataset

In [2]:
df_atlas = pd.read_csv('raw/mundo_onu_adh_municipio (1).csv')

In [3]:
df_lept = pd.read_parquet('processed/df_lept_processed_v1.parquet')

#### Select columns Atlas

In [4]:
columns_atlas = ["id_municipio","expectativa_vida","fecundidade_total","prob_sobrevivencia_40","prob_sobrevivencia_60","taxa_freq_bruta_basico","taxa_freq_bruta_fundamental","taxa_freq_bruta_medio","taxa_freq_liquida_superior","prop_pobreza_extrema","prop_vulner_pobreza","taxa_agua_encanada","taxa_banheiro_agua_encanada","taxa_coleta_lixo","taxa_energia_eletrica","taxa_agua_esgoto_inadequados","taxa_paredes_inadequados","populacao","indice_escolaridade","indice_frequencia_escolar","idhm","idhm_e","idhm_l","idhm_r"]

In [5]:
df_atlas=df_atlas[columns_atlas]

### Select columns Hepa

In [6]:
np.array(df_lept.columns)

array(['TP_NOT', 'ID_AGRAVO', 'DT_NOTIFIC', 'SEM_NOT', 'NU_ANO',
       'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE',
       'DT_SIN_PRI', 'SEM_PRI', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO',
       'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'SG_UF', 'ID_MN_RESI',
       'ID_RG_RESI', 'ID_PAIS', 'NDUPLIC_N', 'DT_DIGITA', 'DT_TRANSUS',
       'DT_TRANSDM', 'DT_TRANSSM', 'DT_TRANSRS', 'DT_TRANSSE',
       'NU_LOTE_V', 'DT_INVEST', 'ID_OCUPA_N', 'ANT_CB_LAM', 'ANT_CB_CRI',
       'ANT_CB_CAI', 'ANT_CB_FOS', 'ANT_CB_SIN', 'ANT_CB_PLA',
       'ANT_CB_COR', 'ANT_CB_ROE', 'ANT_CB_GRA', 'ANT_CB_TER',
       'ANT_CB_LIX', 'ANT_CB_OUT', 'ANT_OU_DES', 'ANT_HUMANO',
       'ANT_ANIMAI', 'CLI_DT_ATE', 'CLI_FEBRE', 'CLI_MIALGI',
       'CLI_CEFALE', 'CLI_PROST', 'CLI_CONGES', 'CLI_PANTUR',
       'CLI_VOMITO', 'CLI_DIARRE', 'CLI_ICTERI', 'CLI_RENAL',
       'CLI_RESPIR', 'CLI_CARDIA', 'CLI_HEMOPU', 'CLI_HEMORR',
       'CLI_MENING', 'CLI_OUTROS', 'CLI_OTRDES', 'ATE_HOSP', 'ATE_DT_INT',
       'ATE_

In [7]:

triage_features = [
    'NU_IDADE_N','CS_SEXO','CS_GESTANT','CS_RACA','CS_ESCOL_N',
    'ID_OCUPA_N','CON_AREA','CON_AMBIEN',
    'ANT_CB_LAM','ANT_CB_CRI','ANT_CB_SIN','ANT_CB_COR','ANT_CB_ROE',
    'ANT_CB_TER','ANT_CB_LIX','ANT_CB_OUT','ANT_ANIMAI','ANT_HUMANO',
    'CLI_FEBRE','CLI_MIALGI','CLI_CEFALE','CLI_PROST','CLI_CONGES',
    'CLI_PANTUR','CLI_VOMITO','CLI_DIARRE','CLI_ICTERI','CLI_RENAL',
    'CLI_RESPIR','CLI_CARDIA','CLI_HEMOPU','CLI_HEMORR','CLI_MENING',
    'CLI_OUTROS','SG_UF','ID_MUNICIP', 'baixa_escolaridade','raca_vulneravel','faixa_vulneravel',
    'vulnerabilidade_social', 'nivel_vulnerabilidade','mes_sintomas','estacao', 'periodo_chuvoso','CLASSI_FIN'
]
df_train=df_lept.loc[:, df_lept.columns.isin(triage_features)]

In [8]:
df_train.columns = [col.lower() for col in df_train.columns ]
df_train['id_municip']=df_train.id_municip.astype(int)
df_train.rename({'id_municip':'id_municipio'},axis=1,inplace=True)

C:\Users\diego\AppData\Local\Temp\ipykernel_21884\3820349990.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['id_municip']=df_train.id_municip.astype(int)
C:\Users\diego\AppData\Local\Temp\ipykernel_21884\3820349990.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train.rename({'id_municip':'id_municipio'},axis=1,inplace=True)


In [9]:
np.array(df_train.columns)

array(['id_municipio', 'nu_idade_n', 'cs_sexo', 'cs_gestant', 'cs_raca',
       'cs_escol_n', 'sg_uf', 'id_ocupa_n', 'ant_cb_lam', 'ant_cb_cri',
       'ant_cb_sin', 'ant_cb_cor', 'ant_cb_roe', 'ant_cb_ter',
       'ant_cb_lix', 'ant_cb_out', 'ant_humano', 'ant_animai',
       'cli_febre', 'cli_mialgi', 'cli_cefale', 'cli_prost', 'cli_conges',
       'cli_pantur', 'cli_vomito', 'cli_diarre', 'cli_icteri',
       'cli_renal', 'cli_respir', 'cli_cardia', 'cli_hemopu',
       'cli_hemorr', 'cli_mening', 'cli_outros', 'classi_fin', 'con_area',
       'con_ambien', 'baixa_escolaridade', 'raca_vulneravel',
       'faixa_vulneravel', 'vulnerabilidade_social',
       'nivel_vulnerabilidade', 'mes_sintomas', 'estacao',
       'periodo_chuvoso'], dtype=object)

In [10]:
df_atlas['id_municipio'] = df_atlas['id_municipio']//10
df_atlas_recent=df_atlas.groupby('id_municipio').tail(1)

In [11]:
df_merge=pd.merge(df_train,df_atlas_recent,on=['id_municipio'],how='inner')

In [12]:
df_merge

,id_municipio,nu_idade_n,cs_sexo,cs_gestant,cs_raca,cs_escol_n,sg_uf,id_ocupa_n,ant_cb_lam,ant_cb_cri,...,taxa_energia_eletrica,taxa_agua_esgoto_inadequados,taxa_paredes_inadequados,populacao,indice_escolaridade,indice_frequencia_escolar,idhm,idhm_e,idhm_l,idhm_r
0,317020,11.0,Masculino,Não se aplica,Não informado,Não informado,MG,Não informado,Não informado,Não informado,...,99.92,0.18,0.50,604013,0.646,0.754,0.789,0.716,0.885,0.776
1,120040,4.0,Masculino,Não se aplica,Parda,10,AC,Não informado,Sim,Não,...,99.34,18.43,5.52,336038,0.607,0.690,0.727,0.661,0.798,0.729
2,431730,50.0,Masculino,Não se aplica,Branca,1ª a 4ª série incompleta do EF,RS,622105,Não,Não,...,99.26,0.80,0.65,30990,0.476,0.659,0.712,0.591,0.861,0.709
3,412550,18.0,Feminino,Não,Branca,Ensino médio completo,PR,421125,Não,Não,...,99.91,0.20,1.74,264210,0.617,0.710,0.758,0.678,0.859,0.749
4,351300,31.0,Masculino,Não se aplica,Branca,Ensino médio incompleto,SP,914405,Não,Não,...,99.82,1.26,1.22,201150,0.639,0.743,0.780,0.707,0.851,0.789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169921,320120,65.0,Masculino,Não se aplica,Branca,Não informado,ES,6220.2,Não,Sim,...,99.80,0.38,0.18,189889,0.561,0.743,0.746,0.677,0.837,0.733
169922,320010,56.0,Feminino,Não,Branca,Não informado,ES,6220.2,Sim,Sim,...,99.97,2.75,0.48,31091,0.366,0.664,0.667,0.544,0.825,0.661
169923,320010,76.0,Masculino,Não se aplica,Branca,Não informado,ES,9999.9,Sim,Sim,...,99.97,2.75,0.48,31091,0.366,0.664,0.667,0.544,0.825,0.661
169924,320520,59.0,Feminino,Não,Parda,Não informado,ES,Não informado,Sim,Não,...,99.92,0.42,0.95,414586,0.717,0.742,0.800,0.734,0.864,0.807


### Merge dados Ocupação Ministério do Trabalho

In [13]:
# Baixe ou clone o repositório datasets-br/cbo e carregue o arquivo
url = "https://raw.githubusercontent.com/datasets-br/cbo/master/data/lista.csv"
cbo = pd.read_csv(url, dtype={"codigo": str})
cbo['codigo'] = cbo['codigo'].apply(lambda x: x.replace('-',''))
#cbo['codigo'] = pd.to_numeric(cbo['codigo'],errors='coerce') 
# Criar dicionário código → termo
mapa_cbo = dict(zip(cbo["codigo"], cbo["termo"]))

def mapear_ocupacao(df, coluna="id_ocupa_n"):
    df = df.copy()
    df[coluna] = df[coluna].map(mapa_cbo)
    df[coluna] =df[coluna].fillna('Nao informado')
    return df


In [14]:
df_merge=mapear_ocupacao(df_merge, coluna="id_ocupa_n")

In [15]:
df_merge

,id_municipio,nu_idade_n,cs_sexo,cs_gestant,cs_raca,cs_escol_n,sg_uf,id_ocupa_n,ant_cb_lam,ant_cb_cri,...,taxa_energia_eletrica,taxa_agua_esgoto_inadequados,taxa_paredes_inadequados,populacao,indice_escolaridade,indice_frequencia_escolar,idhm,idhm_e,idhm_l,idhm_r
0,317020,11.0,Masculino,Não se aplica,Não informado,Não informado,MG,Nao informado,Não informado,Não informado,...,99.92,0.18,0.50,604013,0.646,0.754,0.789,0.716,0.885,0.776
1,120040,4.0,Masculino,Não se aplica,Parda,10,AC,Nao informado,Sim,Não,...,99.34,18.43,5.52,336038,0.607,0.690,0.727,0.661,0.798,0.729
2,431730,50.0,Masculino,Não se aplica,Branca,1ª a 4ª série incompleta do EF,RS,Trabalhador da cultura de arroz,Não,Não,...,99.26,0.80,0.65,30990,0.476,0.659,0.712,0.591,0.861,0.709
3,412550,18.0,Feminino,Não,Branca,Ensino médio completo,PR,Fiscal de caixa,Não,Não,...,99.91,0.20,1.74,264210,0.617,0.710,0.758,0.678,0.859,0.749
4,351300,31.0,Masculino,Não se aplica,Branca,Ensino médio incompleto,SP,Retificador de motores de veículos,Não,Não,...,99.82,1.26,1.22,201150,0.639,0.743,0.780,0.707,0.851,0.789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169921,320120,65.0,Masculino,Não se aplica,Branca,Não informado,ES,Nao informado,Não,Sim,...,99.80,0.38,0.18,189889,0.561,0.743,0.746,0.677,0.837,0.733
169922,320010,56.0,Feminino,Não,Branca,Não informado,ES,Nao informado,Sim,Sim,...,99.97,2.75,0.48,31091,0.366,0.664,0.667,0.544,0.825,0.661
169923,320010,76.0,Masculino,Não se aplica,Branca,Não informado,ES,Nao informado,Sim,Sim,...,99.97,2.75,0.48,31091,0.366,0.664,0.667,0.544,0.825,0.661
169924,320520,59.0,Feminino,Não,Parda,Não informado,ES,Nao informado,Sim,Não,...,99.92,0.42,0.95,414586,0.717,0.742,0.800,0.734,0.864,0.807


Save file

In [16]:
np.array(df_merge.columns)

array(['id_municipio', 'nu_idade_n', 'cs_sexo', 'cs_gestant', 'cs_raca',
       'cs_escol_n', 'sg_uf', 'id_ocupa_n', 'ant_cb_lam', 'ant_cb_cri',
       'ant_cb_sin', 'ant_cb_cor', 'ant_cb_roe', 'ant_cb_ter',
       'ant_cb_lix', 'ant_cb_out', 'ant_humano', 'ant_animai',
       'cli_febre', 'cli_mialgi', 'cli_cefale', 'cli_prost', 'cli_conges',
       'cli_pantur', 'cli_vomito', 'cli_diarre', 'cli_icteri',
       'cli_renal', 'cli_respir', 'cli_cardia', 'cli_hemopu',
       'cli_hemorr', 'cli_mening', 'cli_outros', 'classi_fin', 'con_area',
       'con_ambien', 'baixa_escolaridade', 'raca_vulneravel',
       'faixa_vulneravel', 'vulnerabilidade_social',
       'nivel_vulnerabilidade', 'mes_sintomas', 'estacao',
       'periodo_chuvoso', 'expectativa_vida', 'fecundidade_total',
       'prob_sobrevivencia_40', 'prob_sobrevivencia_60',
       'taxa_freq_bruta_basico', 'taxa_freq_bruta_fundamental',
       'taxa_freq_bruta_medio', 'taxa_freq_liquida_superior',
       'prop_pobreza_extrema',

In [17]:
df_merge.to_parquet('processed/df_lept_atlas.parquet')

In [18]:
df_merge.to_csv('processed/df_lept_atlas.csv',index=False)

In [19]:
cbo.to_csv('raw/lista_cbo.csv')